<a href="https://colab.research.google.com/github/faisalkhan4k/RoomMind/blob/feat%2Fvlm-structured-output/notebooks/00_labelling_images_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Some installations for unsloth

## Install Unsloth and related libraries

In [ ]:
import os
# This bypasses the compilation error by using pre-built wheels
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.14.0" "peft<0.14.0" "accelerate<1.3.0"
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kso_s_to/unsloth_d2c45468fe484d9299c29a5f3ae85823
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kso_s_to/unsloth_d2c45468fe484d9299c29a5f3ae85823
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 128.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 130.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## Upgrade pip and setuptools

In [ ]:
!pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


## Install remaining dependencies

In [ ]:
!pip install -q -U transformers accelerate datasets peft bitsandbytes flash-attn qwen-vl-utils trl

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'flash-attn' when getting requirements to build wheel


## Load Model and  Data

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from datasets import load_dataset
import json
from tqdm import tqdm

# Load Dataset
ds = load_dataset("hammer888/interior_style_dataset", split="train").select(range(1000))

# 1. Define the quantization config (Correct)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load the Teacher Model using the config (Corrected)
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"  # Added for faster processing on Colab T4
)

# 3. Load the Processor (Corrected)
processor = AutoProcessor.from_pretrained(model_id)

# --- PROMPTS ---
SYSTEM_PROMPT = "You are an expert interior designer. Analyze the room image and return ONLY a JSON object."
USER_PROMPT = "Identify furniture items, their style, and estimate a realistic price for each. Return structure: {'items': [{'name': '', 'style': '', 'price': 0}], 'total_estimate': 0}"

def label_image(image):
    # Ensure image is in RGB
    if image.mode != "RGB":
        image = image.convert("RGB")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": USER_PROMPT}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to("cuda")

    # Generate the output
    generated_ids = model.generate(**inputs, max_new_tokens=512)

    # Trim the input tokens from the output
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return output_text.strip()

# Run Labeling
labeled_data = []
for i, sample in enumerate(tqdm(ds)):
    try:
        label = label_image(sample['image'])
        # Store as a dictionary for easier conversion to SFT format later
        labeled_data.append({"image": sample['image'], "label": label})
    except Exception as e:
        print(f"Error at index {i}: {e}")
        continue

# Clean memory before Fine-Tuning
del model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/418 [00:00<?, ?B/s]

data/train-00000-of-00003-78c50153d52a30(…):   0%|          | 0.00/450M [00:00<?, ?B/s]

data/train-00001-of-00003-f71c094e5355c8(…):   0%|          | 0.00/453M [00:00<?, ?B/s]

data/train-00002-of-00003-c0e07c6b6c71e7(…):   0%|          | 0.00/451M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

100%|██████████| 1000/1000 [3:34:28<00:00, 12.87s/it]


## Save Labeled Data to Pickle File

In [ ]:
import pickle
with open('roommind_labels.pkl', 'wb') as f:
    pickle.dump(labeled_data, f)

## Hugging Face Login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# This pulls the token safely from your Colab secrets
hf_token = 'enter hf token'
login(token=hf_token)

## Create and Push Dataset to Hugging Face Hub

In [ ]:
from datasets import Dataset, Features, Image, Value

# 1. Convert your labeled_data list to a HF Dataset
# (Ensures images and JSON strings are paired correctly)
features = Features({
    "image": Image(),
    "label": Value("string")
})

labeled_dataset = Dataset.from_list(labeled_data, features=features)

# 2. Push to your specific ID
# Setting private=True is recommended so others don't see your raw data yet
dataset_id = "mohammedfaisalkhan4000/RoomMind-Interior-Labels"

labeled_dataset.push_to_hub(dataset_id, private=True)

print(f"Success! Your data is at: https://huggingface.co/datasets/{dataset_id}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          |  525kB /  178MB            

Success! Your data is at: https://huggingface.co/datasets/mohammedfaisalkhan4000/RoomMind-Interior-Labels


## Save Labeled Data Locally

In [ ]:
from datasets import Dataset, Features, Image, Value

# 1. Define the schema (features) for your new dataset
features = Features({
    "image": Image(),
    "label": Value("string")
})

# 2. Convert your list to a Hugging Face Dataset object
labeled_dataset = Dataset.from_list(labeled_data, features=features)

# 3. Save it LOCALLY to the Colab disk
labeled_dataset.save_to_disk("roommind_labeled_1k")

# 4. COMPRESS for easy download (Optional but recommended)
!tar -cvf roommind_dataset.tar roommind_labeled_1k

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

roommind_labeled_1k/
roommind_labeled_1k/data-00000-of-00001.arrow
roommind_labeled_1k/dataset_info.json
roommind_labeled_1k/state.json
